# 🐼 Pandas TLDR — Quick Reference
A concise cheat sheet for loading, processing, and evaluating datasets.

---
## Table of Contents
1. [Imports & Setup](#1-imports--setup)
2. [Loading Data](#2-loading-data)
3. [First Look at Your Data](#3-first-look-at-your-data)
4. [Selecting & Filtering](#4-selecting--filtering)
5. [Cleaning Data](#5-cleaning-data)
6. [Feature Engineering](#6-feature-engineering)
7. [Grouping & Aggregation](#7-grouping--aggregation)
8. [Sorting & Ranking](#8-sorting--ranking)
9. [Merging & Joining](#9-merging--joining)
10. [Evaluating a Dataset](#10-evaluating-a-dataset)
11. [Exporting Data](#11-exporting-data)

---
## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np

# Display up to 100 columns, 60 rows, and don't truncate column content
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', None)

---
## 2. Loading Data

In [ ]:
# --- CSV ---
df = pd.read_csv('file.csv')
df = pd.read_csv('file.csv', sep=';')            # Custom delimiter
df = pd.read_csv('file.csv', usecols=['a', 'b']) # Load specific columns only
df = pd.read_csv('file.csv', nrows=1000)         # Load first N rows only
df = pd.read_csv('file.csv', encoding='latin-1') # Fix encoding errors

# --- Excel ---
df = pd.read_excel('file.xlsx')
df = pd.read_excel('file.xlsx', sheet_name='Sheet2')

# --- JSON ---
df = pd.read_json('file.json')

# --- From a URL ---
df = pd.read_csv('https://example.com/data.csv')

# --- From a dict / list ---
df = pd.DataFrame({'col1': [1, 2], 'col2': ['a', 'b']})
df = pd.DataFrame([(1, 'a'), (2, 'b')], columns=['col1', 'col2'])

---
## 3. First Look at Your Data

In [ ]:
df.head()          # First 5 rows
df.tail(10)        # Last 10 rows
df.sample(5)       # 5 random rows

df.shape           # (rows, columns)
df.columns         # Column names
df.dtypes          # Data types per column
df.info()          # Column types + non-null counts

df.describe()                        # Stats for numeric columns
df.describe(include='all')           # Stats for ALL columns (including strings)
df['col'].value_counts()             # Frequency of each unique value
df['col'].value_counts(normalize=True) # As proportions (0.0–1.0)
df['col'].nunique()                  # Number of unique values
df['col'].unique()                   # Array of unique values

---
## 4. Selecting & Filtering

In [ ]:
# --- Selecting columns ---
df['col']                        # Single column → Series
df[['col1', 'col2']]             # Multiple columns → DataFrame

# --- Selecting rows ---
df.loc[0]                        # Row by label/index
df.iloc[0]                       # Row by integer position
df.loc[0:5, 'col1':'col3']       # Rows 0–5, columns col1 to col3 (label-based)
df.iloc[0:5, 0:3]                # Rows 0–4, columns 0–2 (position-based)

# --- Filtering rows by condition ---
df[df['col'] > 10]               # Single condition
df[(df['col'] > 10) & (df['col2'] == 'yes')]  # AND
df[(df['col'] < 5) | (df['col2'] == 'no')]    # OR
df[df['col'].isin(['a', 'b'])]   # Value is in a list
df[~df['col'].isin(['a', 'b'])]  # Value is NOT in a list
df[df['col'].str.contains('keyword', na=False)]  # String contains (safe with NaNs)

# --- Query syntax (cleaner for complex filters) ---
df.query('col > 10 and col2 == "yes"')

---
## 5. Cleaning Data

In [ ]:
# --- Missing values ---
df.isnull().sum()                        # Count nulls per column
df.isnull().mean()                       # Proportion of nulls per column
df.dropna()                              # Drop any row with a null
df.dropna(subset=['col1', 'col2'])       # Drop rows where specific cols are null
df.dropna(thresh=3)                      # Keep rows with at least 3 non-null values
df.fillna(0)                             # Fill all nulls with 0
df['col'].fillna(df['col'].mean())       # Fill nulls with column mean
df['col'].fillna(method='ffill')         # Forward fill (carry last known value)
df['col'].fillna(method='bfill')         # Backward fill

# --- Duplicates ---
df.duplicated().sum()                    # Count duplicate rows
df.drop_duplicates()                     # Drop duplicate rows
df.drop_duplicates(subset=['col'])       # Drop dupes based on one column

# --- Renaming ---
df.rename(columns={'old': 'new'})        # Rename specific columns
df.columns = ['a', 'b', 'c']            # Rename all columns at once
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')  # Normalize names

# --- Dropping ---
df.drop(columns=['col1', 'col2'])        # Drop columns
df.drop(index=[0, 1, 2])                 # Drop rows by index

# --- Type conversion ---
df['col'].astype(int)
df['col'].astype(float)
df['col'].astype(str)
pd.to_datetime(df['date_col'])           # Parse dates
pd.to_numeric(df['col'], errors='coerce') # Coerce non-numeric to NaN

# --- String cleaning ---
df['col'].str.strip()                    # Remove leading/trailing whitespace
df['col'].str.lower()
df['col'].str.upper()
df['col'].str.replace('old', 'new')
df['col'].str.split(',', expand=True)    # Split into multiple columns

# --- Reset index after filtering ---
df.reset_index(drop=True)

---
## 6. Feature Engineering

In [ ]:
# --- New columns ---
df['new_col'] = df['col1'] + df['col2']          # Arithmetic
df['new_col'] = df['col'].apply(lambda x: x * 2) # Apply a function row-by-row
df['label'] = df['score'].apply(lambda x: 'high' if x > 50 else 'low')

# --- apply() with a named function ---
def clean_text(text):
    return text.lower().strip()

df['clean'] = df['text'].apply(clean_text)

# --- map() for simple value substitution ---
df['label'] = df['is_phishing'].map({0: 'safe', 1: 'phishing'})

# --- Binning a numeric column ---
df['bucket'] = pd.cut(df['score'], bins=[0, 30, 70, 100], labels=['low', 'mid', 'high'])

# --- One-hot encoding a categorical column ---
pd.get_dummies(df, columns=['category'])

# --- Working with datetime columns ---
df['date'] = pd.to_datetime(df['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day']   = df['date'].dt.day
df['dow']   = df['date'].dt.day_name()  # e.g. 'Monday'

---
## 7. Grouping & Aggregation

In [ ]:
# --- Basic groupby ---
df.groupby('col')['value'].mean()        # Mean of 'value' per group
df.groupby('col')['value'].sum()
df.groupby('col')['value'].count()

# --- Multiple aggregations at once ---
df.groupby('col').agg({'value': ['mean', 'sum', 'count']})

# --- Named aggregations (cleaner output) ---
df.groupby('col').agg(
    avg_score=('score', 'mean'),
    total=('score', 'sum'),
    count=('score', 'count')
)

# --- Group by multiple columns ---
df.groupby(['col1', 'col2'])['value'].mean()

# --- Pivot table ---
df.pivot_table(values='score', index='label', columns='category', aggfunc='mean')

---
## 8. Sorting & Ranking

In [ ]:
df.sort_values('col')                           # Ascending (default)
df.sort_values('col', ascending=False)          # Descending
df.sort_values(['col1', 'col2'], ascending=[True, False])  # Multi-column sort

df['rank'] = df['score'].rank(ascending=False)  # Rank column (1 = highest)

df.nlargest(10, 'score')   # Top 10 rows by score
df.nsmallest(5, 'score')   # Bottom 5 rows by score

---
## 9. Merging & Joining

In [ ]:
# --- merge() is like SQL JOIN ---
pd.merge(df1, df2, on='id')                     # Inner join (default)
pd.merge(df1, df2, on='id', how='left')         # Left join
pd.merge(df1, df2, on='id', how='outer')        # Full outer join
pd.merge(df1, df2, left_on='id', right_on='ID') # Join on differently-named keys

# --- concat() stacks DataFrames ---
pd.concat([df1, df2])                           # Stack rows (axis=0)
pd.concat([df1, df2], ignore_index=True)        # Reset index after stacking
pd.concat([df1, df2], axis=1)                   # Stack columns side by side

---
## 10. Evaluating a Dataset
Quick checks to understand quality, distribution, and balance before modelling.

In [ ]:
# --- Class balance (critical for classification tasks) ---
df['label'].value_counts()
df['label'].value_counts(normalize=True)  # Shows if dataset is imbalanced

# --- Missing data overview ---
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).sort_values('missing_%', ascending=False)

# --- Correlation matrix (numeric columns) ---
df.corr(numeric_only=True).round(2)

# --- Distribution of a numeric column ---
df['col'].describe()          # Mean, std, min, quartiles, max
df['col'].hist()              # Histogram (requires matplotlib)

# --- Outlier check using IQR ---
Q1 = df['col'].quantile(0.25)
Q3 = df['col'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['col'] < Q1 - 1.5 * IQR) | (df['col'] > Q3 + 1.5 * IQR)]
print(f'{len(outliers)} outliers found')

# --- Cross-tab: relationship between two categorical columns ---
pd.crosstab(df['col1'], df['col2'])
pd.crosstab(df['col1'], df['col2'], normalize='index')  # As row proportions

# --- Memory usage ---
df.memory_usage(deep=True).sum() / 1e6  # Total MB

---
## 11. Exporting Data

In [ ]:
df.to_csv('output.csv', index=False)          # CSV (index=False avoids saving the row index)
df.to_excel('output.xlsx', index=False)       # Excel
df.to_json('output.json', orient='records')   # JSON (one record per row)
df.to_parquet('output.parquet', index=False)  # Parquet (fast, compressed — great for large files)

---
*Tip: chain `.copy()` onto any filtered DataFrame (`df2 = df[df['col'] > 0].copy()`) to avoid `SettingWithCopyWarning` when modifying the result.*